# 真实 attention 几何捕获运行入口

该 Notebook 用于 Colab 冷启动: 先挂载 Google Drive, 再拉取仓库代码、安装依赖、登录 Hugging Face、加载真实 SD3.5 Medium 主模型, 捕获真实运行中的 Q/K 可替代审计 attention map, 生成 attention capture records, 重建 attention geometry 产物, 并把所有核对文件打包保存到 Google Drive 的 SLM 目录。

正式逻辑位于 `paper_workflow/colab_utils/attention_geometry_capture.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 准备 Hugging Face token, 并确认账号已获得 SD3.5 Medium 访问权限。
3. 确认 Google Drive 可挂载。默认保存目录为 `/content/drive/MyDrive/SLM/pilot_paper_results/attention_geometry`。
4. 默认只运行 SD3.5 Medium 主模型, 并要求真实 capture records 不带 unsupported reason。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_PROTOCOL_PROFILE', 'pilot_paper_fixed_fpr_0_01')
os.environ.setdefault('SLM_WM_PROMPT_SET', 'pilot_paper')
os.environ.setdefault('SLM_WM_PROMPT_FILE', 'configs/paper_main_pilot_paper_prompts.txt')
os.environ.setdefault('SLM_WM_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/pilot_paper_results/attention_geometry')
os.environ.setdefault('SLM_WM_ATTENTION_CAPTURE_COUNT', '4')
os.environ.setdefault('SLM_WM_ATTENTION_TOKEN_COUNT', '16')
print({
    'protocol_profile': os.environ['SLM_WM_PROTOCOL_PROFILE'],
    'prompt_set': os.environ['SLM_WM_PROMPT_SET'],
    'prompt_file': os.environ['SLM_WM_PROMPT_FILE'],
    'drive_output_dir': os.environ['SLM_WM_DRIVE_OUTPUT_DIR'],
    'attention_capture_count': os.environ['SLM_WM_ATTENTION_CAPTURE_COUNT'],
    'attention_token_count': os.environ['SLM_WM_ATTENTION_TOKEN_COUNT'],
})


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub


In [ ]:
import importlib
import importlib.metadata as importlib_metadata
import importlib.util

# 只做只读依赖诊断, 不清理 sys.modules, 避免在同一 Colab 内核中重载 numpy / torch。
importlib.invalidate_caches()

import diffusers
import transformers
from diffusers import StableDiffusion3Pipeline
from transformers import CLIPTextModelWithProjection
from transformers.generation import GenerationMixin

dependency_report = {
    'diffusers': diffusers.__version__,
    'transformers': transformers.__version__,
    'numpy': importlib_metadata.version('numpy'),
    'tokenizers': importlib_metadata.version('tokenizers'),
    'accelerate': importlib_metadata.version('accelerate'),
    'huggingface_hub': importlib_metadata.version('huggingface_hub'),
    'scipy_available': importlib.util.find_spec('scipy') is not None,
    'torchvision_available': importlib.util.find_spec('torchvision') is not None,
    'stable_diffusion_pipeline': StableDiffusion3Pipeline.__name__,
    'clip_text_model_with_projection': CLIPTextModelWithProjection.__name__,
    'generation_mixin': GenerationMixin.__name__,
}
print(dependency_report)


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
from paper_workflow.colab_utils.attention_geometry_capture import run_default_attention_geometry_plan

attention_geometry_summary = run_default_attention_geometry_plan(root='.')
assert attention_geometry_summary['attention_geometry_ready'] is True, '真实 attention geometry 未就绪'
attention_geometry_summary


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.attention_geometry_capture import package_attention_geometry_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ.get('SLM_WM_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/pilot_paper_results/attention_geometry')
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'attention_geometry_package_{utc_suffix}_{short_commit}.zip'
archive_record = package_attention_geometry_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/real_attention_geometry').glob('*.json')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8'))

for result_path in sorted(Path('outputs/attention_geometry').glob('*.json')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8'))
